# Payment instrument cleaning

1. Load the raw dataset found in this folder (`.csv` preferred, otherwise `.xlsx`/`.xls`).
2. Show the distinct values of `payment_instrument`.
3. Add a new column: values containing `,` or `/` become `'Multiple'`; all others keep their original value. Final values are title-cased (e.g. `Credit card` → `Credit Card`).
4. Save the final dataframe as CSV (same base name) to `r-models/data/`.

All logic lives in the `PaymentInstrumentCleaner` class so it can be reused/imported later.

In [20]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

In [21]:
class PaymentInstrumentCleaner:
    """Reusable pipeline: load -> inspect -> derive column -> export."""

    RAW_EXTENSIONS = (".csv", ".xlsx", ".xls")

    def __init__(
        self,
        data_dir: str | Path | None = None,
        output_dir: str | Path | None = None,
        source_column: str = "payment_instrument",
        new_column: str = "payment_instrument_group",
        multiple_label: str = "Multiple",
        separators: str = ",/",
    ) -> None:
        self.data_dir = Path(data_dir).resolve() if data_dir else self._default_data_dir()
        self.output_dir = (
            Path(output_dir).resolve() if output_dir else self._repo_root() / "r-models" / "data"
        )
        self.source_column = source_column
        self.new_column = new_column
        self.multiple_label = multiple_label
        self._sep_pattern = "[" + re.escape(separators) + "]"
        self.input_path: Path | None = None
        self.df: pd.DataFrame | None = None

    # ---------- path helpers ----------
    @staticmethod
    def _default_data_dir() -> Path:
        cwd = Path.cwd().resolve()
        if cwd.name == "data-cleaning":
            return cwd
        if (cwd / "data-cleaning").is_dir():
            return cwd / "data-cleaning"
        return cwd

    def _repo_root(self) -> Path:
        """Walk up from data_dir until a folder containing `r-models` is found."""
        for parent in (self.data_dir, *self.data_dir.parents):
            if (parent / "r-models").is_dir():
                return parent
        return self.data_dir.parent

    @staticmethod
    def _normalize(name: str) -> str:
        return re.sub(r"[\s_\-]+", "_", str(name).strip().lower())

    def _match_column(self, columns) -> str | None:
        target = self._normalize(self.source_column)
        return next((c for c in columns if self._normalize(c) == target), None)

    # ---------- step 1: load ----------
    def find_input_file(self) -> Path:
        files = [
            p for p in sorted(self.data_dir.iterdir())
            if p.suffix.lower() in self.RAW_EXTENSIONS and not p.name.startswith(("~$", "."))
        ]
        if not files:
            raise FileNotFoundError(f"No CSV/Excel file found in {self.data_dir}")
        csvs = [p for p in files if p.suffix.lower() == ".csv"]
        return (csvs or files)[0]

    def load(self, path: str | Path | None = None, sheet_name: str | None = None) -> pd.DataFrame:
        self.input_path = Path(path).resolve() if path else self.find_input_file()
        if self.input_path.suffix.lower() == ".csv":
            df = pd.read_csv(self.input_path, low_memory=False)
        else:
            sheet_name = sheet_name or self._find_sheet_with_column(self.input_path)
            df = pd.read_excel(self.input_path, sheet_name=sheet_name)
        col = self._match_column(df.columns)
        if col is None:
            raise KeyError(f"Column '{self.source_column}' not found in {self.input_path.name}")
        self.source_column = col
        self.df = df
        print(f"Loaded {len(df):,} rows x {df.shape[1]} cols from {self.input_path.name}"
              + (f" [sheet: {sheet_name}]" if sheet_name else ""))
        return df

    def _find_sheet_with_column(self, path: Path) -> str:
        with pd.ExcelFile(path) as xls:
            for sheet in xls.sheet_names:
                header = pd.read_excel(xls, sheet_name=sheet, nrows=0)
                if self._match_column(header.columns):
                    return sheet
        raise KeyError(f"No sheet in {path.name} contains '{self.source_column}'")

    def _require_df(self) -> pd.DataFrame:
        if self.df is None:
            raise RuntimeError("Call load() first.")
        return self.df

    # ---------- step 2: inspect ----------
    def distinct_values(self, column: str | None = None) -> pd.DataFrame:
        df = self._require_df()
        col = column or self.source_column
        return (
            df[col].value_counts(dropna=False)
            .rename_axis(col).reset_index(name="count")
        )

    # ---------- step 3: transform ----------
    def add_group_column(self) -> pd.DataFrame:
        df = self._require_df()
        values = df[self.source_column]
        is_multiple = values.astype("string").str.contains(self._sep_pattern, regex=True, na=False)
        grouped = values.where(~is_multiple, self.multiple_label)
        df[self.new_column] = self._title_case(grouped)
        return df

    @staticmethod
    def _title_case(values: pd.Series) -> pd.Series:
        """Capitalize each word so variants like 'Credit card' / 'Credit Card' collapse."""
        return values.astype("string").str.strip().str.title()

    # ---------- step 4: export ----------
    def export(self, filename: str | None = None) -> Path:
        df = self._require_df()
        self.output_dir.mkdir(parents=True, exist_ok=True)
        name = filename or f"{self.input_path.stem}.csv"
        out_path = self.output_dir / name
        df.to_csv(out_path, index=False)
        print(f"Wrote {len(df):,} rows to {out_path}")
        return out_path

    # ---------- convenience ----------
    def run(self, path: str | Path | None = None, sheet_name: str | None = None) -> Path:
        self.load(path, sheet_name)
        self.add_group_column()
        return self.export()

## 1. Load data

In [22]:
cleaner = PaymentInstrumentCleaner()
df = cleaner.load()
df.head()

Loaded 12,397 rows x 42 cols from rpw_dataset_2011_2025_q1.xlsx [sheet: Dataset (from Q2 2016)]


,id,period,source_code,source_name,source_region,source_income,source_lending,source_G8G20,destination_code,destination_name,...,cc2 total cost %,inter lcu bank fx,transparent,Standard Note,note2,receiving network coverage,pickup location,pickup method,date,corridor
0,620163173,2016_2Q,USA,United States,..,High income,..,G8/G20,BRA,Brazil,...,2.44,3.46,yes,NaN,NaN,Medium,NaN,Bank account,11/May/2016,USABRA
1,620163176,2016_2Q,USA,United States,..,High income,..,G8/G20,BRA,Brazil,...,1.87,3.46,yes,Bank account payment,NaN,Medium,NaN,Bank account,11/May/2016,USABRA
2,620163177,2016_2Q,USA,United States,..,High income,..,G8/G20,BRA,Brazil,...,1.87,3.46,yes,Bank account payment,NaN,Medium,NaN,Bank account,11/May/2016,USABRA
3,620163179,2016_2Q,USA,United States,..,High income,..,G8/G20,BRA,Brazil,...,2.47,3.46,yes,NaN,NaN,High,Agent,Cash,11/May/2016,USABRA
4,620163180,2016_2Q,USA,United States,..,High income,..,G8/G20,BRA,Brazil,...,6.04,3.46,yes,Card payment,NaN,Medium,NaN,Bank account,11/May/2016,USABRA


## 2. Distinct values of `payment_instrument`

In [23]:
cleaner.distinct_values()

,payment instrument,count
0,Cash,3780
1,Bank account transfer,3076
2,Debit card,1897
3,Debit/credit card,1360
4,Credit Card,1320
5,"Bank account transfer,Cash",302
6,"Bank account transfer,Debit/credit card",211
7,"Credit Card,Debit card",165
8,Credit card,96
9,"Cash,Debit/credit card",92


## 3. Derive the grouped column (`,` or `/` → `multiple`)

In [25]:
df = cleaner.add_group_column()
cleaner.distinct_values(cleaner.new_column)

,payment_instrument_group,count
0,Cash,3780
1,Bank Account Transfer,3076
2,Multiple,2227
3,Debit Card,1897
4,Credit Card,1416
5,Door To Door,1


## 4. Export to `r-models/data`

In [26]:
output_path = cleaner.export()
output_path

Wrote 12,397 rows to /Users/yonnyleyvasuarez/Documents/GitHub/isye-7406-g105/r-models/data/rpw_dataset_2011_2025_q1.csv


PosixPath('/Users/yonnyleyvasuarez/Documents/GitHub/isye-7406-g105/r-models/data/rpw_dataset_2011_2025_q1.csv')